In [ ]:
# Setup: repo path and single-paper config (with optional env overrides).
import os
import sys
from pathlib import Path

repo_root = Path.cwd().resolve()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
sys.path.append(str(repo_root))

PAPER_FILENAME = os.getenv("PAPER_FILENAME", "2.md")
DOMAIN_NAME = os.getenv("DOMAIN_NAME", "domain_1")

print(f"Using paper: {PAPER_FILENAME}")
print(f"Using domain: {DOMAIN_NAME}")

Using paper: 2.md
Using domain: domain_1


In [ ]:
# Imports: two prompt builders + LLM call helper.
import json

from domain_shot import build_question_scoring_prompt as qs
from domain_shot import build_tree_guided_prompt as tq
from domain_shot.evaluation import DEFAULT_MODEL, call_llm_and_process

In [ ]:
# Load input texts for one paper and one domain.
paper_path = Path(tq.DEFAULT_PAPERS_DIR) / PAPER_FILENAME
paper_text = tq.read_text_file(str(paper_path))
intro_text = tq.load_default_guidance_intro()
domain_questions_text = tq.load_domain_questions(DOMAIN_NAME)
decision_tree_text = tq.load_domain_decision_tree(DOMAIN_NAME)

print(f"Paper: {paper_path.name}")
print(f"Domain: {DOMAIN_NAME}")

FileNotFoundError: Missing decision-tree file for domain 'domain_1'. Expected one of: /home/user/Thesis/conservation-llm-critical-appraisal/data/decision_trees_only_md/domain_1_decision_tree.md, /home/user/Thesis/conservation-llm-critical-appraisal/data/decision_trees_only_md/domain_1.md

In [ ]:
# Build both prompt message payloads.
tree_messages = tq.build_prompt_messages(
    domain_name=DOMAIN_NAME,
    intro_text=intro_text,
    decision_tree_text=decision_tree_text,
    domain_questions_text=domain_questions_text,
    paper_text=paper_text,
 )

scoring_messages = qs.build_prompt_messages(
    domain_name=DOMAIN_NAME,
    intro_text=intro_text,
    domain_questions_text=domain_questions_text,
    paper_text=paper_text,
 )

print("Built tree-guided and question-scoring prompt messages.")

In [ ]:
# Run both prompts on the same single paper.
tree_result = call_llm_and_process(DOMAIN_NAME, tree_messages, DEFAULT_MODEL)
scoring_result = call_llm_and_process(DOMAIN_NAME, scoring_messages, DEFAULT_MODEL)

print("Tree-guided response keys:", list(tree_result.keys()))
print("Question-scoring response keys:", list(scoring_result.keys()))

In [ ]:
# Save one combined output JSON.
output_dir = repo_root / "output" / "single_paper_two_prompts_notebook"
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / f"{paper_path.stem}_{DOMAIN_NAME}.json"

combined_results = {
    "paper": PAPER_FILENAME,
    "domain": DOMAIN_NAME,
    "tree_guided": tree_result,
    "question_scoring": scoring_result,
}

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(combined_results, f, indent=2, ensure_ascii=False)

print(f"Saved: {output_path}")